# Use Case 4 - Glue ETL Job Notebook

## Converted from Glue Python job to Jupyter notebook

This notebook was generated from the original Glue ETL Python script to make the logic easier to teach and run step by step in a notebook.

### Use case
ETL for monitoring, bias review support, and model lifecycle control.

### ETL purpose
Read baseline and incoming scoring data, apply monitoring-oriented ETL steps, compare profiles, and publish review-ready artifacts for lifecycle decisions.

### How to teach this notebook
- Start with configuration and paths
- Run extraction first
- Inspect transformation logic
- Validate outputs before publish
- Explain how the same logic runs as a repeatable Glue job in production


## Notebook guidance

When running this in a notebook:
- replace AWS placeholders as needed
- inspect DataFrames after key transforms
- connect each step back to ETL principles: extract, transform, validate, load/publish


## Step 1 - Imports

Import Glue and PySpark libraries. `pyspark.sql.types` is included so we can safely cast columns after reading CSVs where all columns default to `StringType`.


In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType


## Step 2 - Initialise Glue context and resolve job arguments

This job takes three path arguments:
- `INPUT_PATH` — incoming scoring batch (the data the deployed model just scored)
- `BASELINE_PATH` — the training baseline used during original model development
- `OUTPUT_PATH` — destination prefix for monitoring artifacts


In [ ]:
args = getResolvedOptions(sys.argv, ['JOB_NAME', 'INPUT_PATH', 'BASELINE_PATH', 'OUTPUT_PATH'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)


## Step 3 - Extract: read baseline and incoming batch from S3

Both datasets are read with `inferSchema` enabled. The baseline was written by the UC3 Glue job and already has correct column types. The incoming batch may arrive with schema differences — a row count check surfaces that early.


In [ ]:
incoming = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(args['INPUT_PATH'])
)
baseline = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(args['BASELINE_PATH'])
)
print(f"Baseline rows : {baseline.count()}  cols : {len(baseline.columns)}")
print(f"Incoming rows : {incoming.count()}  cols : {len(incoming.columns)}")


## Step 4 - Transform: build numeric-only mean profiles

Aggregate each dataset into a single-row mean profile across numeric columns only. Categorical columns are excluded here because `F.mean` on a `StringType` column raises a Spark `AnalysisException` at runtime.

**Why numeric only?** Mean-shift is the primary statistical drift signal for continuous features. Categorical drift is handled in the comparison step using mode (most frequent value) comparison instead.


In [ ]:
from pyspark.sql.types import NumericType

# Identify numeric columns only - applying F.mean to StringType columns raises AnalysisException
def numeric_cols(df, exclude=None):
    exclude = set(exclude or [])
    return [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, NumericType) and f.name not in exclude
    ]

baseline_num_cols = numeric_cols(baseline, exclude=['customerID'])
incoming_num_cols = numeric_cols(incoming, exclude=['customerID'])

# Single-row mean profile for each dataset
baseline_profile = baseline.select([F.mean(c).alias(c) for c in baseline_num_cols])
incoming_profile = incoming.select([F.mean(c).alias(c) for c in incoming_num_cols])

baseline_profile.show()


## Step 5 - Compare profiles and flag drift

Compare the incoming mean against the baseline mean for each numeric column. A percentage shift greater than 10% is flagged as a potential drift signal. These findings drive the model lifecycle decision ladder.

**Teaching point:** this is the monitoring equivalent of the validation step in UC3 — both are data quality gates, just at different points in the model lifecycle.


In [ ]:
# Collect profiles to driver for comparison (single-row aggregates are safe to collect)
baseline_row = baseline_profile.first().asDict() if baseline_profile.count() > 0 else {}
incoming_row  = incoming_profile.first().asDict() if incoming_profile.count() > 0 else {}

# Common numeric columns present in both datasets
common_cols = [c for c in baseline_num_cols if c in incoming_num_cols]

findings = []
for col in common_cols:
    b_val = baseline_row.get(col)
    i_val = incoming_row.get(col)
    if b_val is not None and b_val != 0:
        pct_change = (i_val - b_val) / b_val
        flagged = abs(pct_change) > 0.10
    else:
        pct_change = None
        flagged = False
    findings.append({
        'column_name': col,
        'check_type': 'mean_shift',
        'baseline_mean': b_val,
        'incoming_mean': i_val,
        'pct_change': round(pct_change, 4) if pct_change is not None else None,
        'flag': flagged
    })

findings_df = spark.createDataFrame(findings)
flagged_df  = findings_df.filter(F.col('flag') == True)

print(f"Columns checked: {len(findings)}  Flagged: {flagged_df.count()}")
flagged_df.show(truncate=False)


## Step 6 - Load: write profiles and findings to S3, then commit

Publish three artifacts:
1. `baseline_profile_json` — reference profile for this job run
2. `incoming_profile_json` — profile of the new scoring batch
3. `drift_findings` — all columns with their drift metrics, used by dashboards or downstream workflows
4. `flagged_drift_findings` — only columns that exceeded the drift threshold, used to trigger review alerts

`job.commit()` finalises the Glue job bookmark so re-runs only process new data.


In [ ]:
baseline_profile.write.mode('overwrite').json(args['OUTPUT_PATH'] + '/baseline_profile_json')
incoming_profile.write.mode('overwrite').json(args['OUTPUT_PATH'] + '/incoming_profile_json')
findings_df.write.mode('overwrite').option('header', True).csv(args['OUTPUT_PATH'] + '/drift_findings')
flagged_df.write.mode('overwrite').option('header', True).csv(args['OUTPUT_PATH'] + '/flagged_drift_findings')
job.commit()
